# Foundation contact springs via `node_to_surface` + `zeroLength`

`node_to_surface` was designed to bridge a 6-DOF master to a 3-DOF
solid face. It generates a phantom node for every slave, rigid-links
the master to the phantoms, and equal-DOFs each phantom back to its
original slave in translations.

The last step — `equalDOF` — is just *one* way to consume the
phantom↔slave pair. If you swap that emission for a `zeroLength`
element, the pair becomes a **spring** between a fixed phantom
(pinned to ground via the rigid-link chain) and the free structural
node above it. That gives you a whole distributed Winkler bed with
no extra bookkeeping — the resolver already matched every slave to
a co-located phantom.

In this notebook we build a rigid concrete footing on an elastic
soil bed. The soil is modelled as one vertical `zeroLength` spring
per bottom-face mesh node, with an `ENT` (Elastic–No–Tension)
material so nodes lift off under tension instead of pulling the
footing down. Loading is vertical dead weight + overturning moment,
pushed past the footing's kern so you can see the uplift.

### The per-record dispatch trick

We deliberately use **two** `node_to_surface` records:

1. `ground_ref → bottom_face`: swapped to `zeroLength`/`ENT` springs.
2. `top_ref → top_face`: kept as standard `equalDOF`, used to push
   the load as force + moment through a single top reference node.

We pick which treatment to apply by iterating the **compound**
iterator `.node_to_surfaces()` and checking `rec.master_node` — a
direct use of the "atomic vs compound" convention the earlier tutorial
talks about.

In [1]:
from apeGmsh import apeGmsh, Results
import numpy as np

# ---- Geometry [mm] ------------------------------------------------
L  = 1000.0    # footing length  (x)
W  = 1000.0    # footing width   (y)
H  = 200.0     # footing height  (z)
lc = 200.0

# ---- Footing (concrete) material ----------------------------------
E_c  = 30_000.0   # MPa
nu_c = 0.2

# ---- Soil spring ---------------------------------------------------
k_v  = 10_000.0   # N/mm per bottom-face spring (vertical only)

# ---- Loading -------------------------------------------------------
P_v  = -100_000.0     # N  (negative z = downward dead weight)
M_y  =  4.0e7         # N*mm  (overturning moment about Y)

ecc  = abs(M_y / P_v)
kern = L / 6.0
print(f'eccentricity  e = |M/P| = {ecc:.0f} mm')
print(f'footing kern  L/6       = {kern:.0f} mm '
      f'\u2192 e > kern means uplift expected')

eccentricity  e = |M/P| = 400 mm
footing kern  L/6       = 167 mm → e > kern means uplift expected


## 1. Build the model

A single box for the footing, two reference points, two physical
groups for the top/bottom faces, and two `node_to_surface` couplings.
Nothing exotic — the interesting work happens in the OpenSees
emission, not in the model definition.

In [2]:
m = apeGmsh(model_name='footing_contact', verbose=False)
m.begin()

# Footing block, centred on the X/Y axes, base at z=0.
m.model.geometry.add_box(-L/2, -W/2, 0, L, W, H, label='footing')
m.model.selection.select_volumes().to_physical('pg_footing')

# Reference points — master of each node_to_surface coupling.
m.model.geometry.add_point(0, 0, 0, lc=lc, label='ground_ref')
m.model.geometry.add_point(0, 0, H, lc=lc, label='top_ref')

# Bottom (contact) and top (loading) faces, picked by thin slab so
# the four side faces are not accidentally pulled in.
INF = max(L, W) * 2
m.model.selection.select_surfaces(
    in_box=(-INF, -INF, -0.1, INF, INF, 0.1)).to_physical('bottom_face')
m.model.selection.select_surfaces(
    in_box=(-INF, -INF, H - 0.1, INF, INF, H + 0.1)).to_physical('top_face')

# Two node_to_surface couplings — same resolver, different downstream
# treatment in the OpenSees build below.
m.constraints.node_to_surface('ground_ref', 'bottom_face')
m.constraints.node_to_surface('top_ref',    'top_face')

# Load applied at top_ref: vertical dead weight + overturning moment.
m.loads.point(
    target='top_ref',
    force_xyz=[0, 0, P_v],
    moment_xyz=[0, M_y, 0],
)

m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=3)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'total nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')

total nodes: 190
  line2  n=64
  tri3   n=372
  tet4   n=467
  point1 n=10


## 2. Two compound records, two downstream uses

The resolver produced **two** `NodeToSurfaceRecord` objects: one for
the bottom (soil contact) and one for the top (load application).
We look them up by `master_node` later to decide whether to emit
`equalDOF` or `zeroLength`.

In [3]:
nc = fem.nodes.constraints
print(f'{len(nc)} compound record(s)\n')

for rec in nc.node_to_surfaces():
    print(f'  name            : {rec.name}')
    print(f'  master node     : {rec.master_node}')
    print(f'  n phantoms      : {len(rec.phantom_nodes)}')
    print(f'  rigid_beam pairs: {len(rec.rigid_link_records)}')
    print(f'  equal_dof pairs : {len(rec.equal_dof_records)}')
    print()

ground_master = int(fem.nodes.get_ids(target='ground_ref')[0])
top_master    = int(fem.nodes.get_ids(target='top_ref')[0])
print(f'ground_ref master id: {ground_master}')
print(f'top_ref    master id: {top_master}')

2 compound record(s)

  name            : ground_ref → bottom_face
  master node     : 9
  n phantoms      : 74
  rigid_beam pairs: 74
  equal_dof pairs : 74

  name            : top_ref → top_face
  master node     : 10
  n phantoms      : 74
  rigid_beam pairs: 74
  equal_dof pairs : 74

ground_ref master id: 9
top_ref    master id: 10


## 3. Build the OpenSees model

There is one subtlety: `zeroLength` requires both ends to have the
**same** `ndf`. The default phantom setup leaves phantoms at `ndf=6`
(so a `rigidLink('beam')` can latch onto them), but solid tet nodes
are `ndf=3`. Instead of adding an intermediate bridge node, we
simply route the two records differently at emission time:

* **Ground-coupling phantoms**: emitted as `ndf=3` and `fix`-ed in
  all three translations. They act as pinned ground anchors directly.
  We skip `rigid_link_records` for this record entirely — a fully
  fixed node is already pinned, the rigid chain adds nothing.
* **Top-coupling phantoms**: standard `ndf=6`, rigid-linked to
  `top_ref`, `equalDOF`-ed to top-face slaves.

Both phantom sets come from the same resolver call — we just iterate
`.node_to_surfaces()` and pick a per-record strategy. The spring
emission loop then knows the ground phantoms already share `ndf=3`
with their slaves, so `zeroLength` is legal.

In [4]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

# --- Solid tet nodes (ndf=3 override) ------------------------------
n_solid = 0
for nid, xyz in fem.nodes.get(target='pg_footing'):
    ops.node(nid, *xyz, '-ndf', 3)
    n_solid += 1

# --- Reference points --------------------------------------------
# top_ref receives the applied load and is the master of the top
# coupling's rigid link chain.
# ground_ref is a vestigial placeholder — the bottom coupling uses
# fixed phantoms directly, so ground_ref plays no role in the
# kinematics. We still emit it (fully fixed) so every id in
# fem.nodes.ids has a node on the OpenSees side.
for nid, xyz in fem.nodes.get(target='top_ref'):
    ops.node(nid, *xyz)
for nid, xyz in fem.nodes.get(target='ground_ref'):
    ops.node(nid, *xyz)
    ops.fix(int(nid), 1, 1, 1, 1, 1, 1)

# --- Phantom nodes, routed per-record -----------------------------
# Ground phantoms → ndf=3, fully fixed (direct ground anchors)
# Top phantoms    → ndf=6 (rigid-link target of top_ref)
n_ground_phantoms = 0
n_top_phantoms    = 0
for rec in fem.nodes.constraints.node_to_surfaces():
    if rec.master_node == ground_master:
        for pid, pxyz in zip(rec.phantom_nodes, rec.phantom_coords):
            pid = int(pid)
            ops.node(pid, *pxyz, '-ndf', 3)
            ops.fix(pid, 1, 1, 1)
            n_ground_phantoms += 1
    else:
        for pid, pxyz in zip(rec.phantom_nodes, rec.phantom_coords):
            ops.node(int(pid), *pxyz)
            n_top_phantoms += 1

# Lock the footing's horizontal rigid-body modes. The vertical-only
# springs do not resist ux/uy; pin those DOFs on every bottom-face
# slave node. uz is still free, so the spring can work.
base_ids = [int(n) for n in fem.nodes.get_ids(target='bottom_face')]
for nid in base_ids:
    ops.fix(nid, 1, 1, 0)

print(f'solid nodes   (ndf=3): {n_solid}')
print(f'ground phantoms (ndf=3, pinned): {n_ground_phantoms}')
print(f'top phantoms    (ndf=6):         {n_top_phantoms}')
print(f'base nodes pinned horizontally: {len(base_ids)}')

solid nodes   (ndf=3): 188
ground phantoms (ndf=3, pinned): 74
top phantoms    (ndf=6):         74
base nodes pinned horizontally: 74


In [5]:
# Concrete solid.
mat_tag = 1
ops.nDMaterial('ElasticIsotropic', mat_tag, E_c, nu_c)

max_eid = 0
n_tets = 0
for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', eid, *conn, mat_tag)
        max_eid = max(max_eid, int(eid))
        n_tets += 1

# Soil spring material — ENT (Elastic No-Tension) with a ~10%
# Elastic stabilizer in parallel. Pure ENT has zero tangent on the
# tension branch, which makes Newton oscillate at the compression/
# tension transition and the initial stiffness matrix singular.
# A parallel Elastic gives a non-zero tangent everywhere; 10% is
# enough for clean Newton convergence while still showing clear
# uplift (lifted nodes carry ~10% of nominal stress, a factor of 10
# below the compression side).
stab_frac    = 0.10
ent_raw_tag  = 10
ent_stab_tag = 11
ent_tag      = 2
ops.uniaxialMaterial('ENT',      ent_raw_tag,  k_v)
ops.uniaxialMaterial('Elastic',  ent_stab_tag, k_v * stab_frac)
ops.uniaxialMaterial('Parallel', ent_tag, ent_raw_tag, ent_stab_tag)

print(f'tet4 elements emitted: {n_tets}')
print(f'spring material tag {ent_tag} = '
      f'Parallel(ENT({k_v:.0f}), Elastic({k_v*stab_frac:.0f}))')

tet4 elements emitted: 467
spring material tag 2 = Parallel(ENT(10000), Elastic(1000))


In [6]:
# Per-record dispatch — this is the core trick of the notebook.
next_eid = max_eid + 1
n_rl = 0
n_zl = 0
n_ed = 0

for rec in fem.nodes.constraints.node_to_surfaces():
    if rec.master_node == ground_master:
        # Soil contact: one vertical zeroLength spring per phantom-slave
        # pair. Phantoms are already ndf=3 and fixed, so no rigid link
        # is needed — the phantom IS the ground anchor.
        for pair in rec.equal_dof_records:
            ops.element(
                'zeroLength', next_eid,
                pair.master_node,   # phantom (ndf=3, fixed)
                pair.slave_node,    # solid face node (ndf=3, free in uz)
                '-mat', ent_tag,
                '-dir', 3,
            )
            next_eid += 1
            n_zl += 1
    else:
        # Top: standard rigid_link + equalDOF for load transfer.
        for pair in rec.rigid_link_records:
            ops.rigidLink('beam', pair.master_node, pair.slave_node)
            n_rl += 1
        for pair in rec.equal_dof_records:
            ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)
            n_ed += 1

print(f'rigid links (top)    : {n_rl}')
print(f'zeroLength springs   : {n_zl}  (soil contact, ENT)')
print(f'equalDOF pairs (top) : {n_ed}')

rigid links (top)    : 74
zeroLength springs   : 74  (soil contact, ENT)
equalDOF pairs (top) : 74


In [7]:
# Load pattern.
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)
print(f'loads applied: {sum(1 for _ in fem.nodes.loads.by_kind("nodal"))}')

# Nonlinear static — ENT creates contact nonlinearity as springs
# transition between compression and (near-)tension. Advance the load
# in 20 sub-steps so Newton can track each transition cleanly.
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-5, 200)
ops.algorithm('Newton')
n_steps = 20
ops.integrator('LoadControl', 1.0 / n_steps)
ops.analysis('Static')

for step in range(n_steps):
    ok = ops.analyze(1)
    assert ok == 0, f'analysis failed at step {step + 1}: {ok}'
print(f'analysis converged in {n_steps} steps')

top_disp = ops.nodeDisp(top_master)
print(f'\ntop_ref node {top_master}:')
print(f'  uz = {top_disp[2]:+.5e} mm')
print(f'  Ry = {top_disp[4]:+.5e} rad')

loads applied: 1


analysis converged in 20 steps

top_ref node 10:
  uz = -4.62423e-02 mm
  Ry = +7.65115e-04 rad


## 4. Uplift check and linear reference

For a **linear** bed of the same vertical stiffness (no uplift
cut-off), the rigid-footing solution is

$$
u_{\text{avg}} = \frac{P_v}{N\,k_v},
\qquad
\theta = \frac{M_y}{k_v\,\sum_i x_i^{2}}
$$

With `e = |M/P| = 400 mm` we are well past the kern (`L/6 = 167 mm`)
so the ENT contact law should bite hard: the linear solution predicts
large positive `uz` on the tension side, which becomes genuine lift-off
once the no-tension material removes that reaction. We expect:

* **More rotation** than the linear prediction (fewer springs resist `M_y`)
* **A non-trivial number of nodes** with positive `uz` (lifted off
  from the ground phantoms)
* **A contact centroid** that has shifted away from the footing centre
  toward the compression edge

We verify the first two quantitatively and visualise the third with
the deformed-shape viewer below.

In [8]:
# Linear Winkler reference (for comparison only — no contact cutoff).
bot_coords = fem.nodes.get_coords(target='bottom_face')
N          = bot_coords.shape[0]
sum_x2     = float((bot_coords[:, 0] ** 2).sum())

u_avg_lin  = P_v / (N * k_v)           # negative = settlement
theta_lin  = M_y / (k_v * sum_x2)

print('linear Winkler reference (no uplift cutoff):')
print(f'  N springs   = {N}')
print(f'  Sum(x^2)    = {sum_x2:.3e} mm^2')
print(f'  u_avg_lin   = {u_avg_lin:+.4e} mm')
print(f'  theta_lin   = {theta_lin:+.4e} rad')
print()
print('FEM with ENT soil:')
print(f'  top_ref uz  = {top_disp[2]:+.4e} mm')
print(f'  top_ref Ry  = {top_disp[4]:+.4e} rad')
print(f'  rotation amplification: '
      f'Ry_FEM / theta_lin = {top_disp[4] / theta_lin:.3f}')

# Examine bottom-face kinematics directly.
bot_ids = list(int(n) for n in fem.nodes.get_ids(target='bottom_face'))
uz_bot  = np.array([ops.nodeDisp(n)[2] for n in bot_ids])
xs_bot  = bot_coords[:, 0]

n_contact = int((uz_bot < -1e-6).sum())
n_lifted  = int((uz_bot > +1e-6).sum())

print()
print('Bottom-face node kinematics:')
print(f'  in contact (uz < 0): {n_contact} / {N}')
print(f'  lifted off (uz > 0): {n_lifted} / {N}')
print(f'  uz range: [{uz_bot.min():+.4e}, {uz_bot.max():+.4e}] mm')

# Where is the contact/lift-off boundary in x?
if n_contact and n_lifted:
    x_edge_lift = float(xs_bot[uz_bot > 0].min())
    x_edge_comp = float(xs_bot[uz_bot < 0].max())
    # Rough contact centroid — weight by compressive displacement.
    comp_mask  = uz_bot < 0
    x_cg_comp  = float(
        (xs_bot[comp_mask] * uz_bot[comp_mask]).sum() /
        uz_bot[comp_mask].sum()
    )
    print(f'  first lifted node at   x = {x_edge_lift:+7.0f} mm')
    print(f'  max compressed node at x = {x_edge_comp:+7.0f} mm')
    print(f'  compression centroid   x = {x_cg_comp:+7.0f} mm  '
          f'(footing centre = 0, kern edge = {L/6:+.0f})')

linear Winkler reference (no uplift cutoff):
  N springs   = 74
  Sum(x^2)    = 7.769e+06 mm^2
  u_avg_lin   = -1.3514e-01 mm
  theta_lin   = +5.1487e-04 rad

FEM with ENT soil:
  top_ref uz  = -4.6242e-02 mm
  top_ref Ry  = +7.6511e-04 rad
  rotation amplification: Ry_FEM / theta_lin = 1.486

Bottom-face node kinematics:
  in contact (uz < 0): 40 / 74
  lifted off (uz > 0): 34 / 74
  uz range: [-4.2644e-01, +3.3657e-01] mm
  first lifted node at   x =    -500 mm
  max compressed node at x =    +500 mm
  compression centroid   x =    +352 mm  (footing centre = 0, kern edge = +167)


## 5. Deformed view

Push the nodal displacement field into `Results.from_fem` and launch
the apeGmsh viewer. Toggle **Show Deformed** to see the tilted
footing — the compression edge settles, the uplift edge separates
from its phantom. With a large deformation scale, the "lift off" side
should be clearly visible as positive `Uz`.

## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [11]:
from apeGmsh import workdir
OUT = workdir()
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = OUT / "capture.h5"
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 7 ---
    import openseespy.opensees as ops

    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)
    ops.timeSeries('Linear', 1)

    # --- Solid tet nodes (ndf=3 override) ------------------------------
    n_solid = 0
    for nid, xyz in fem.nodes.get(target='pg_footing'):
        ops.node(nid, *xyz, '-ndf', 3)
        n_solid += 1

    # --- Reference points --------------------------------------------
    # top_ref receives the applied load and is the master of the top
    # coupling's rigid link chain.
    # ground_ref is a vestigial placeholder — the bottom coupling uses
    # fixed phantoms directly, so ground_ref plays no role in the
    # kinematics. We still emit it (fully fixed) so every id in
    # fem.nodes.ids has a node on the OpenSees side.
    for nid, xyz in fem.nodes.get(target='top_ref'):
        ops.node(nid, *xyz)
    for nid, xyz in fem.nodes.get(target='ground_ref'):
        ops.node(nid, *xyz)
        ops.fix(int(nid), 1, 1, 1, 1, 1, 1)

    # --- Phantom nodes, routed per-record -----------------------------
    # Ground phantoms → ndf=3, fully fixed (direct ground anchors)
    # Top phantoms    → ndf=6 (rigid-link target of top_ref)
    n_ground_phantoms = 0
    n_top_phantoms    = 0
    for rec in fem.nodes.constraints.node_to_surfaces():
        if rec.master_node == ground_master:
            for pid, pxyz in zip(rec.phantom_nodes, rec.phantom_coords):
                pid = int(pid)
                ops.node(pid, *pxyz, '-ndf', 3)
                ops.fix(pid, 1, 1, 1)
                n_ground_phantoms += 1
        else:
            for pid, pxyz in zip(rec.phantom_nodes, rec.phantom_coords):
                ops.node(int(pid), *pxyz)
                n_top_phantoms += 1

    # Lock the footing's horizontal rigid-body modes. The vertical-only
    # springs do not resist ux/uy; pin those DOFs on every bottom-face
    # slave node. uz is still free, so the spring can work.
    base_ids = [int(n) for n in fem.nodes.get_ids(target='bottom_face')]
    for nid in base_ids:
        ops.fix(nid, 1, 1, 0)

    print(f'solid nodes   (ndf=3): {n_solid}')
    print(f'ground phantoms (ndf=3, pinned): {n_ground_phantoms}')
    print(f'top phantoms    (ndf=6):         {n_top_phantoms}')
    print(f'base nodes pinned horizontally: {len(base_ids)}')
    # --- copied from cell 8 ---
    # Concrete solid.
    mat_tag = 1
    ops.nDMaterial('ElasticIsotropic', mat_tag, E_c, nu_c)

    max_eid = 0
    n_tets = 0
    for group in fem.elements.get(element_type='tet4'):
        for eid, conn in group:
            ops.element('FourNodeTetrahedron', eid, *conn, mat_tag)
            max_eid = max(max_eid, int(eid))
            n_tets += 1

    # Soil spring material — ENT (Elastic No-Tension) with a ~10%
    # Elastic stabilizer in parallel. Pure ENT has zero tangent on the
    # tension branch, which makes Newton oscillate at the compression/
    # tension transition and the initial stiffness matrix singular.
    # A parallel Elastic gives a non-zero tangent everywhere; 10% is
    # enough for clean Newton convergence while still showing clear
    # uplift (lifted nodes carry ~10% of nominal stress, a factor of 10
    # below the compression side).
    stab_frac    = 0.10
    ent_raw_tag  = 10
    ent_stab_tag = 11
    ent_tag      = 2
    ops.uniaxialMaterial('ENT',      ent_raw_tag,  k_v)
    ops.uniaxialMaterial('Elastic',  ent_stab_tag, k_v * stab_frac)
    ops.uniaxialMaterial('Parallel', ent_tag, ent_raw_tag, ent_stab_tag)

    print(f'tet4 elements emitted: {n_tets}')
    print(f'spring material tag {ent_tag} = '
          f'Parallel(ENT({k_v:.0f}), Elastic({k_v*stab_frac:.0f}))')
    # --- copied from cell 9 ---
    # Per-record dispatch — this is the core trick of the notebook.
    next_eid = max_eid + 1
    n_rl = 0
    n_zl = 0
    n_ed = 0

    for rec in fem.nodes.constraints.node_to_surfaces():
        if rec.master_node == ground_master:
            # Soil contact: one vertical zeroLength spring per phantom-slave
            # pair. Phantoms are already ndf=3 and fixed, so no rigid link
            # is needed — the phantom IS the ground anchor.
            for pair in rec.equal_dof_records:
                ops.element(
                    'zeroLength', next_eid,
                    pair.master_node,   # phantom (ndf=3, fixed)
                    pair.slave_node,    # solid face node (ndf=3, free in uz)
                    '-mat', ent_tag,
                    '-dir', 3,
                )
                next_eid += 1
                n_zl += 1
        else:
            # Top: standard rigid_link + equalDOF for load transfer.
            for pair in rec.rigid_link_records:
                ops.rigidLink('beam', pair.master_node, pair.slave_node)
                n_rl += 1
            for pair in rec.equal_dof_records:
                ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)
                n_ed += 1

    print(f'rigid links (top)    : {n_rl}')
    print(f'zeroLength springs   : {n_zl}  (soil contact, ENT)')
    print(f'equalDOF pairs (top) : {n_ed}')
    # --- copied from cell 10 ---
    # Load pattern.
    ops.pattern('Plain', 1, 1)
    for load in fem.nodes.loads.by_kind('nodal'):
        fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
        mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
        ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)
    print(f'loads applied: {sum(1 for _ in fem.nodes.loads.by_kind("nodal"))}')

    # Nonlinear static — ENT creates contact nonlinearity as springs
    # transition between compression and (near-)tension. Advance the load
    # in 20 sub-steps so Newton can track each transition cleanly.
    ops.constraints('Penalty', 1e15, 1e15)
    ops.numberer('RCM')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1.0e-5, 200)
    ops.algorithm('Newton')
    n_steps = 20
    ops.integrator('LoadControl', 1.0 / n_steps)
    ops.analysis('Static')

    for step in range(n_steps):
        ok = ops.analyze(1)
        cap.step(t=ops.getTime())
        assert ok == 0, f'analysis failed at step {step + 1}: {ok}'
    print(f'analysis converged in {n_steps} steps')

    top_disp = ops.nodeDisp(top_master)
    print(f'\ntop_ref node {top_master}:')
    print(f'  uz = {top_disp[2]:+.5e} mm')
    print(f'  Ry = {top_disp[4]:+.5e} rad')
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


solid nodes   (ndf=3): 188
ground phantoms (ndf=3, pinned): 74
top phantoms    (ndf=6):         74
base nodes pinned horizontally: 74
tet4 elements emitted: 467
spring material tag 2 = Parallel(ENT(10000), Elastic(1000))
rigid links (top)    : 74
zeroLength springs   : 74  (soil contact, ENT)
equalDOF pairs (top) : 74
loads applied: 1


analysis converged in 20 steps

top_ref node 10:
  uz = -4.62423e-02 mm
  Ry = +7.65115e-04 rad
capture -> example_footing_contact_springs_capture.h5 (321.4 KB)


In [12]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
from apeGmsh.results import Results
results = Results.from_native(results_capture)
results.viewer(blocking=False)


[skip viewer] APEGMSH_SKIP_VIEWER set
